# 02 — Correlation Analysis

**Primary:** Tianyi Qin  
**Support:** Tuan Wei

This notebook uses the exact processed rows, target and split created in `01_preprocessing.ipynb`.

Designed variable set:
- property size: `accommodates`, `bedrooms`, `bathrooms`;
- location: `distance_cbd_km`;
- amenities: `amenity_count`;
- target: `high_price`.

All four required methods are computed for every unique pair: **Pearson, Spearman, Mutual Information (MI), and Normalised Mutual Information (NMI)**.

## 1. Imports and processed data

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Run notebooks/01_preprocessing.ipynb first to create processed_listings.csv."
    )

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target counts:", df["high_price"].value_counts().sort_index().to_dict())

## 2. Variable set and method implementation

Pearson and Spearman are computed directly on numeric representations.

For MI/NMI, continuous/count variables are discretised into up to five quantile bins. The binary target is left as binary. This makes the pairwise MI/NMI calculation symmetric and reproducible; the report should explicitly state this implementation choice.

In [ ]:
VARIABLES = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "high_price",
]

missing_cols = [c for c in VARIABLES if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing correlation columns: {missing_cols}")

VARIABLE_TYPES = {
    "accommodates": "numeric",
    "bedrooms": "numeric",
    "bathrooms": "numeric",
    "distance_cbd_km": "numeric",
    "amenity_count": "numeric",
    "high_price": "binary",
}

display(df[VARIABLES].describe())

## 3. Helpers for MI/NMI discretisation

In [ ]:
def to_information_categories(series, variable_type, max_bins=5):
    if variable_type == "binary":
        return pd.to_numeric(series, errors="coerce")

    numeric = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    valid = numeric.notna()

    if valid.sum() < 2:
        return out

    # qcut may drop duplicate edges for low-cardinality numeric variables.
    binned = pd.qcut(
        numeric.loc[valid],
        q=min(max_bins, numeric.loc[valid].nunique()),
        labels=False,
        duplicates="drop",
    )
    out.loc[valid] = binned.astype(float)
    return out

## 4. Compute all four methods for every pair

In [ ]:
rows = []

for a, b in combinations(VARIABLES, 2):
    pair = df[[a, b]].dropna().copy()

    pearson = pearsonr(pair[a], pair[b]).statistic
    spearman = spearmanr(pair[a], pair[b]).statistic

    a_disc = to_information_categories(
        pair[a], VARIABLE_TYPES[a]
    )
    b_disc = to_information_categories(
        pair[b], VARIABLE_TYPES[b]
    )
    valid = a_disc.notna() & b_disc.notna()

    mi = mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )
    nmi = normalized_mutual_info_score(
        a_disc.loc[valid].astype(int),
        b_disc.loc[valid].astype(int),
    )

    rows.append({
        "var_a": a,
        "var_b": b,
        "n": len(pair),
        "pearson": float(pearson),
        "spearman": float(spearman),
        "mutual_information": float(mi),
        "normalised_mutual_information": float(nmi),
        "implementation_note": (
            "Pearson/Spearman on numeric values; MI/NMI on 5-quantile "
            "discretisation for numeric variables; high_price left binary."
        ),
    })

corr_results = pd.DataFrame(rows)
display(corr_results)

## 5. Target associations

This table isolates every predictor–target pair so the group can identify the strongest/weakest associations with actual values.

In [ ]:
target_rows = corr_results[
    (corr_results["var_a"] == "high_price")
    | (corr_results["var_b"] == "high_price")
].copy()

target_rows["predictor"] = np.where(
    target_rows["var_a"] == "high_price",
    target_rows["var_b"],
    target_rows["var_a"],
)
target_rows["abs_pearson"] = target_rows["pearson"].abs()
target_rows["abs_spearman"] = target_rows["spearman"].abs()

display(
    target_rows[
        [
            "predictor",
            "n",
            "pearson",
            "spearman",
            "mutual_information",
            "normalised_mutual_information",
        ]
    ].sort_values("normalised_mutual_information", ascending=False)
)

## 6. Predictor–predictor relationships

This is used to identify potential redundancy/multicollinearity before modelling.

In [ ]:
predictor_pairs = corr_results[
    (corr_results["var_a"] != "high_price")
    & (corr_results["var_b"] != "high_price")
].copy()

predictor_pairs["abs_pearson"] = predictor_pairs["pearson"].abs()
predictor_pairs["abs_spearman"] = predictor_pairs["spearman"].abs()

display(
    predictor_pairs.sort_values(
        ["abs_pearson", "normalised_mutual_information"],
        ascending=False,
    )
)

## 7. Correlation matrices and figures

In [ ]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

corr_results.to_csv(TABLE_OUT / "correlation_results.csv", index=False)
target_rows.to_csv(TABLE_OUT / "correlation_target_associations.csv", index=False)
predictor_pairs.to_csv(TABLE_OUT / "correlation_predictor_pairs.csv", index=False)

def symmetric_matrix(results, value_col):
    matrix = pd.DataFrame(
        np.eye(len(VARIABLES)),
        index=VARIABLES,
        columns=VARIABLES,
        dtype=float,
    )

    # MI has a meaningful non-one diagonal, but the diagonal is not used
    # in pairwise interpretation. Keep 1.0 visually for consistency.
    for _, row in results.iterrows():
        matrix.loc[row["var_a"], row["var_b"]] = row[value_col]
        matrix.loc[row["var_b"], row["var_a"]] = row[value_col]
    return matrix

for method, col in {
    "pearson": "pearson",
    "spearman": "spearman",
    "mi": "mutual_information",
    "nmi": "normalised_mutual_information",
}.items():
    matrix = symmetric_matrix(corr_results, col)
    matrix.to_csv(TABLE_OUT / f"{method}_matrix.csv")

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.values, aspect="auto")
    ax.set_xticks(range(len(VARIABLES)))
    ax.set_yticks(range(len(VARIABLES)))
    ax.set_xticklabels(VARIABLES, rotation=45, ha="right")
    ax.set_yticklabels(VARIABLES)
    ax.set_title(f"{method.upper()} pairwise matrix")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_OUT / f"correlation_{method}.png", dpi=200)
    plt.close(fig)

print("Saved correlation tables to:", TABLE_OUT)
print("Saved correlation figures to:", FIG_OUT)

## 8. Evidence checklist for group-written interpretation

Use the generated tables to write the report yourselves. Before finalising this section, identify with actual values:
- which methods agree and which diverge;
- the strongest, weakest and near-absent predictor–target relationships;
- any strong predictor–predictor relationships;
- one concrete downstream modelling/feature decision informed by the correlation results;
- plausible confounders/biases, using association rather than causal language.